# Model the Air Quality Index with a gaussian process.

In [1]:
import os
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from MuyGPyS._test.sampler import print_results
from MuyGPyS.neighbors import NN_Wrapper
from MuyGPyS.gp import MuyGPS
from MuyGPyS.gp.deformation import Isotropy, l2, Anisotropy
from MuyGPyS.gp.hyperparameter import AnalyticScale, Parameter, VectorParameter
from MuyGPyS.gp.kernels import Matern
from MuyGPyS.gp.noise import HomoscedasticNoise
from MuyGPyS.optimize import Bayes_optimize, L_BFGS_B_optimize
from MuyGPyS.optimize.batch import sample_batch
from MuyGPyS.optimize.loss import lool_fn, mse_fn, pseudo_huber_fn, looph_fn
import pandas as pd
import time

# Import and setup data

In [2]:
BASE_DIR = os.path.abspath("..") + "\\"
aqi_data = np.genfromtxt(BASE_DIR + "data\\biggest_day_aqi_coords_scaled_noAH.csv", delimiter=',', skip_header=1)
#print(aqi_data[:10])

In [3]:
#randomly split the data into features and responces, and those in turn into training and testing paritions.
aqi_x, aqi_y = aqi_data[:, :-1], aqi_data[:, -1]
aqi_x_train, aqi_x_test, aqi_y_train, aqi_y_test = train_test_split(aqi_x, aqi_y, test_size=0.1, random_state=20)

#use the mean of the training points to provide basic de-meaning of the points.
aqi_y_mean = aqi_y_train.mean()
aqi_y_train = aqi_y_train - aqi_y_mean

In [4]:
kfold = KFold(n_splits=10, shuffle=True, random_state=24)

# Nearest Neighbor and Batches

In [5]:
#setup nearest nieghbors
nn_count = 30
nbrs_lookup = NN_Wrapper(aqi_x_train, nn_count, nn_method="exact", algorithm="ball_tree")


In [6]:
#for the sake of making use of batches, will use a batch value of 1500. this will include all of the data points.
batch_count = 1500
#get the indices of the training points and the indices of each points neighbors.
batch_indices, batch_nn_indices = sample_batch(
    nbrs_lookup, batch_count, len(aqi_x_train)
)

# setting and optimizing hyperparamters

In [7]:
#create our unoptimized muyGP object. set the ranges we want the hyperparameters to be optimized between.
aqi_muygps = MuyGPS(
    kernel=Matern(
        smoothness=Parameter("log_sample", (0.5, 2.5)),
        deformation=Isotropy(
            l2,
            length_scale= Parameter("log_sample", (0.01, 0.20)),
        ),
    ),
    noise=HomoscedasticNoise("sample", (0.003, 0.25)),
    scale=AnalyticScale(),
)

In [8]:
#use the indices of the training points, and of the training points neighbors, together with the points' coordinates and values
#to calculate the crosswise distances between points and the pairwise distances between the neighbors
#and also the actual value of each point, and each point's neighbor.
(
    batch_crosswise_dists,
    batch_pairwise_dists,
    batch_ys,
    batch_nn_ys,
) = aqi_muygps.make_train_tensors(
    batch_indices,
    batch_nn_indices,
    aqi_x_train,
    aqi_y_train,
)

In [9]:
optimize_start = time.time()
#optimize the parameters for 15 random initilization runs, and then 25 optimizing runs.
aqi_muygps_optimized = Bayes_optimize(
    aqi_muygps,
    batch_ys,
    batch_nn_ys,
    batch_crosswise_dists,
    batch_pairwise_dists,
    loss_fn=lool_fn,
    verbose=True,
    random_state=24,
    init_points=20,
    n_iter=30,
)

# aqi_muygps_optimized = L_BFGS_B_optimize(
#     aqi_muygps,
#     batch_ys,
#     batch_nn_ys,
#     batch_crosswise_dists,
#     batch_pairwise_dists,
#     loss_fn=mse_fn,
#     verbose=True,
# )
optimize_end = time.time()
optimize_length = optimize_end - optimize_start
print(f"Gaussian Process optimization took {optimize_length} seconds")

parameters to be optimized: ['length_scale', 'smoothness', 'noise']
bounds: [[0.01  0.2  ]
 [0.5   2.5  ]
 [0.003 0.25 ]]
initial x0: [0.03517863 0.94941689 0.00498307]
|   iter    |  target   | length... | smooth... |   noise   |
-------------------------------------------------------------
| 1         | -7221.099 | 0.0351786 | 0.9494168 | 0.0049830 |
| 2         | -6350.749 | 0.1924032 | 1.8990240 | 0.2499672 |
| 3         | -6444.558 | 0.0518127 | 1.2221127 | 0.1857407 |
| 4         | -6199.853 | 0.1993265 | 1.1326939 | 0.0367265 |
| 5         | -6296.004 | 0.0829562 | 1.1410385 | 0.0935044 |
| 6         | -6229.267 | 0.1448337 | 2.3002848 | 0.1349265 |
| 7         | -6488.619 | 0.0569858 | 1.8436131 | 0.1417470 |
| 8         | -6434.340 | 0.1130863 | 2.2868952 | 0.2111665 |
| 9         | -6500.916 | 0.0681423 | 1.7623395 | 0.1710189 |
| 10        | -6311.286 | 0.1943812 | 2.2871343 | 0.2357791 |
| 11        | -6177.703 | 0.1320228 | 1.7292952 | 0.0592377 |
| 12        | -6472.878 |

In [10]:
#also optimize the scale
aqi_muygps_optimized = aqi_muygps_optimized.optimize_scale(
    batch_pairwise_dists,
    batch_nn_ys
)

# Inference

In [11]:
test_count = aqi_x_test.shape[0]
#get the indices of all the test points
test_indices = np.arange(test_count)
#get the indices of the neighbors for each of those test points as well
test_nn_indices, _ = nbrs_lookup.get_nns(aqi_x_test)

In [12]:
#create tesors to store the crosswise distances and pairwise distances between points and the point's neighbors.
#as well as the actual values for each points neighbors.
(
    test_crosswise_dists,
    test_pairwise_dists,
    test_nn_ys,
) = aqi_muygps.make_predict_tensors(
    test_indices,
    test_nn_indices,
    aqi_x_test,
    aqi_x_train,
    aqi_y_train,
)

In [13]:
#calculate kernels to store the crosswise covariance and pairwise covariance. these are used to know how much a points neighbors influence it.
kcross = aqi_muygps_optimized.kernel(test_crosswise_dists)
kin = aqi_muygps_optimized.kernel(test_pairwise_dists)

In [14]:
#Take the crosswise covariance and pairwise covariance to know how the neighbors effect each point,
#and then use the values of the points to calculate the actual value based on the neighbors influence.
predictions = aqi_muygps_optimized.posterior_mean(kin, kcross, test_nn_ys)
#because mean was removed from the datapoints before, need to add it back in.
predictions = predictions + aqi_y_mean
#calculate the variances, how much the guess is spread, meaning uncertainty.
variances = aqi_muygps_optimized.posterior_variance(kin, kcross)
#get the range that values need to be in in order to fall within 95% certainty.
confidence_intervals = np.sqrt(variances) * 1.96
#coverage is the proportion of guesses that differ from the true response by no more than the confidence interval size.
coverage = np.count_nonzero(np.abs(aqi_y_test - predictions) < confidence_intervals) / test_count

In [15]:
#print the results of the model
print_results(
    aqi_y_test, ("optimized", aqi_muygps_optimized, predictions, variances, confidence_intervals, coverage)
)

name,smoothness,length scale,noise variance,variance scale,rmse,mean variance,mean confidence interval,coverage
optimized,0.500000,0.200000,0.082581,996.432827,13.671413,79.112961,16.968275,0.825688


In [16]:
print(predictions[:10])
print(variances[:10])

[35.29447376 46.51269668 40.20565337 52.50517166 46.0150268  51.89219132
 65.35523522 31.31654363 46.12951653 45.55343598]
[169.40876311  55.15264735  43.1159066   77.14957587  96.56553286
  39.19821269  89.99064619  78.98532146 179.32979348  48.47744707]


In [17]:
predictions_df = pd.DataFrame(predictions)
predictions_df.describe()

,0
count,109.000000
mean,47.599992
std,11.103753
min,12.658322
25%,41.328177
50%,47.756631
75%,55.333874
max,71.393081
